# T10 - Evaluer la baseline des bookmakers

**Objectif** : Evaluer la qualite des probabilites issues des cotes bet365 avant de developper un modele predictif propre.

**Perimetre** : Saison 2024-2025, Ligue 1 francaise. Bookmaker : bet365 uniquement. Metriques : log loss, Brier score, calibration.

**Limites** : Une seule saison, un seul bookmaker.

In [ ]:
import os
import warnings
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sqlalchemy
from sqlalchemy import text

matplotlib.rcParams.update({"figure.figsize": (12, 7), "figure.dpi": 100, "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12})
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
IMG_DIR = ROOT / "notebooks"
IMG_DIR.mkdir(parents=True, exist_ok=True)

print("OK imports")
print(f"Racine: {ROOT}")

In [ ]:
db_user = os.getenv("MARIADB_USER", "football")
db_password = os.getenv("MARIADB_PASSWORD", "football")
db_host = os.getenv("MARIADB_HOST", "localhost")
db_port = os.getenv("MARIADB_PORT", "3306")
db_name = os.getenv("MARIADB_DATABASE", "football_odds")
engine = sqlalchemy.create_engine(f"mariadb+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")
print(f"Connecte a {db_name}")

---

## 1. Observations admissibles

Extraction des matchs bet365 avec cotes de fermeture valides (3 issues + result connu).

In [ ]:
sql_query = """
SELECT
    match_id, source_match_key, competition, season, match_date,
    home_team, away_team, home_goals, away_goals, result_1n2,
    bookmaker, odds_type,
    home_odds, draw_odds, away_odds,
    prob_home_raw, prob_draw_raw, prob_away_raw,
    sum_probs_raw, bookmaker_margin,
    prob_home_norm, prob_draw_norm, prob_away_norm
FROM v_implicit_probabilities
WHERE bookmaker = "bet365"
  AND odds_type = "closing"
  AND home_odds IS NOT NULL
  AND draw_odds IS NOT NULL
  AND away_odds IS NOT NULL
  AND result_1n2 IS NOT NULL
ORDER BY match_id
"""

In [ ]:
with engine.connect() as conn:
    df = conn.execute(text(sql_query)).mappings()
    df = pd.DataFrame(list(df))
print(f"Matchs charges: {len(df)}")

In [ ]:
decimal_cols = ["home_odds", "draw_odds", "away_odds", "prob_home_raw", "prob_draw_raw", "prob_away_raw", "sum_probs_raw", "bookmaker_margin", "prob_home_norm", "prob_draw_norm", "prob_away_norm"]
for col in decimal_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"Matchs admissibles: {len(df)}")
print(df.describe().round(4))

---

## 2. Vecteurs de probabilites 1N2

- Brutes : 1/cote (somme > 1)
- Normalisees : somme = 1
- Issue reelle encodee en one-hot

In [ ]:
df["prob_home_norm"] = pd.to_numeric(df["prob_home_norm"], errors="coerce")
df["prob_draw_norm"] = pd.to_numeric(df["prob_draw_norm"], errors="coerce")
df["prob_away_norm"] = pd.to_numeric(df["prob_away_norm"], errors="coerce")
df["sum_norm"] = df["prob_home_norm"] + df["prob_draw_norm"] + df["prob_away_norm"]
smin = df["sum_norm"].min()
smax = df["sum_norm"].max()
print(f"Somme norm - min: {smin:.6f}, max: {smax:.6f}")

probs_norm = df[["prob_home_norm", "prob_draw_norm", "prob_away_norm"]].values
probs_raw = df[["prob_home_raw", "prob_draw_raw", "prob_away_raw"]].values
y_encoded = pd.get_dummies(df["result_1n2"], dtype=int)[["H", "D", "A"]].values
print(f"Shape: {probs_norm.shape}, H={y_encoded[:,0].sum()}, D={y_encoded[:,1].sum()}, A={y_encoded[:,2].sum()}")

---

## 3. Metriques : log loss et Brier score

Log loss : -1/N * Sum(y * log(p))
Brier : 1/N * Sum((y - p)^2)

Plus bas = meilleur. Brier de 0 = parfait.

In [ ]:
def compute_log_loss(y_true, y_pred, epsilon=1e-15):
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]

def compute_brier(y_true, y_pred):
    return np.mean(np.sum((y_true - y_pred) ** 2, axis=1))

ll_norm = compute_log_loss(y_encoded, probs_norm)
br_norm = compute_brier(y_encoded, probs_norm)
ll_raw = compute_log_loss(y_encoded, probs_raw)
br_raw = compute_brier(y_encoded, probs_raw)

print("=" * 50)
print("METRIQUES BASELINE BET365")
print("=" * 50)
print(f"Log Loss  : brutes={ll_raw:.4f}, normees={ll_norm:.4f}")
print(f"Brier     : brutes={br_raw:.4f}, normees={br_norm:.4f}")

In [ ]:
random_probs = np.full((len(df), 3), 1/3)
ll_random = compute_log_loss(y_encoded, random_probs)
br_random = compute_brier(y_encoded, random_probs)
print(f"Reference aleatoire: log_loss={ll_random:.4f}, brier={br_random:.4f}")

---

## 4. Calibration

Tranches de probabilites vs frequences observees.

In [ ]:
def compute_calibration(y_true, probs, n_bins=10):
    calib = {}
    for k, cls_name in enumerate(["H", "D", "A"]):
        cp = probs[:, k]
        ca = y_true[:, k]
        edges = np.linspace(0, 1, n_bins + 1)
        mid, obs, cnt = [], [], []
        for i in range(n_bins):
            m = (cp >= edges[i]) & (cp < edges[i+1])
            if i == n_bins - 1:
                m = (cp >= edges[i]) & (cp <= edges[i+1])
            if m.sum() > 0:
                mid.append(float(cp[m].mean()))
                obs.append(float(ca[m].mean()))
                cnt.append(int(m.sum()))
        if mid:
            calib[cls_name] = {"mid": mid, "obs": obs, "cnt": cnt}
    return calib

calib_norm = compute_calibration(y_encoded, probs_norm, n_bins=10)
classes = {"H": "Victoire Domicile", "D": "Match Nul", "A": "Victoire Extérieur"}

In [ ]:
for cls in ["H", "D", "A"]:
    print(f"=== {classes[cls]} ({cls}) ===")
    d = calib_norm[cls]
    for i in range(len(d["mid"])):
        p = d["mid"][i]; o = d["obs"][i]; n = d["cnt"][i]; g = o - p
        print(f"  bin~{p:.1f}: pred={p:.4f} obs={o:.4f} n={n} ecart={g:+.4f}")

---

## 5. Visualisation calibration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"H": "#2ecc71", "D": "#e74c3c", "A": "#3498db"}

for ax, cls in zip(axes, ["H", "D", "A"]):
    d = calib_norm[cls]
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Ideal")
    ax.scatter(d["mid"], d["obs"], color=colors[cls], s=100, label=classes[cls])
    for i in range(len(d["mid"])):
        txt = "n=" + str(d["cnt"][i])
        ax.annotate(txt, xy=(d["mid"][i], d["obs"][i]), xytext=(5,5), fontsize=8)
    ax.set_xlabel("Prob predite")
    ax.set_ylabel("Freq observee")
    ax.set_title(f"Calibration - {classes[cls]}")
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.2)

plt.suptitle("Calibration bet365 (cotes de fermeture)", fontsize=14, y=1.02)
plt.tight_layout()
path = IMG_DIR / "img_calibration_normalisees.png"
plt.savefig(path, bbox_inches="tight")
print(f"Sauvegarde: {path}")
plt.show()

---

## 6. Distribution de la marge bet365

In [ ]:
margins = df["bookmaker_margin"].values
print(f"Marge bet365 - Saison 2024-2025")
print(f"Moyenne: {margins.mean()*100:.2f}%")
print(f"Mediane: {np.median(margins)*100:.2f}%")
print(f"Std: {margins.std()*100:.2f}%")
print(f"Min: {margins.min()*100:.2f}%, Max: {margins.max()*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(margins*100, bins=20, color="#9b59b6", edgecolor="white", alpha=0.8)
axes[0].axvline(margins.mean()*100, color="red", ls="--", lw=2)
axes[0].axvline(np.median(margins)*100, color="orange", ls="--", lw=2)
axes[0].set_xlabel("Marge (%)"); axes[0].set_ylabel("Matchs")
axes[0].set_title("Distribution de la marge bet365")
axes[0].grid(True, alpha=0.2)

axes[1].boxplot(margins*100, vert=True, patch_artist=True, widths=0.3)
axes[1].set_ylabel("Marge bet365 (%)")
axes[1].set_title("Boxplot Marge")
axes[1].grid(True, alpha=0.2)

plt.suptitle("Marge implicite bet365", fontsize=14, y=1.02)
plt.tight_layout()
path = IMG_DIR / "img_distribution_marge.png"
plt.savefig(path, bbox_inches="tight")
print(f"Sauvegarde: {path}")
plt.show()

---

## 7. Analyse par tranche de cote

In [ ]:
def analyze_by_odds(df, tc="H", oc="home_odds"):
    ci = {"H":0,"D":1,"A":2}[tc]
    be = [0, 1.5, 2.0, 3.0, 5.0, float("inf")]
    bl = ["[0-1.5[","[1.5-2.0[","[2.0-3.0[","[3.0-5.0[","[5.0+]"]
    res = []
    for i,(lo,hi) in enumerate(zip(be[:-1],be[1:])):
        m = (df[oc]>=lo) & (df[oc]<hi)
        if m.sum()==0: continue
        pp = probs_norm[m,ci].mean()
        of_ = y_encoded[m,ci].mean()
        res.append({"T":bl[i],"N":int(m.sum()),"pred":pp,"obs":of_,"ecart":of_-pp})
    return pd.DataFrame(res)

for c,cl,oc in [("H","Victoire Domicile","home_odds"),("D","Match Nul","draw_odds"),("A","Victoire Extérieur","away_odds")]:
    print(f"=== {cl} ({c}) ===")
    r = analyze_by_odds(df, c, oc)
    print(r.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = {"H": "#2ecc71", "D": "#e74c3c", "A": "#3498db"}
xo = {"H": -0.3, "D": 0, "A": 0.3}
tranches = ["[0-1.5[","[1.5-2.0[","[2.0-3.0[","[3.0-5.0[","[5.0+]"]
x = np.arange(len(tranches))

for c,cl,oc in [("H","Victoire Domicile","home_odds"),("D","Match Nul","draw_odds"),("A","Victoire Extérieur","away_odds")]:
    r = analyze_by_odds(df, c, oc)
    heights = np.full(len(tranches), np.nan)
    for idx, t in enumerate(tranches):
        match = r[r["T"]==t]
        if len(match)>0:
            heights[idx] = match["ecart"].values[0]
    ax.bar(x+xo[c], heights, width=0.25, label=cl, color=colors[c], alpha=0.8)

ax.axhline(y=0, color="black", ls="-", lw=0.8)
ax.set_xlabel("Tranche de cote")
ax.set_ylabel("Ecart (pp)")
ax.set_title("Ecart calibration bet365 par tranche de cote")
ax.set_xticks(x); ax.set_xticklabels(tranches); ax.legend()
ax.grid(True, alpha=0.2, axis="y")

for c,cl,oc in [("H","Victoire Domicile","home_odds"),("D","Match Nul","draw_odds"),("A","Victoire Extérieur","away_odds")]:
    r = analyze_by_odds(df, c, oc)
    for idx, t in enumerate(tranches):
        match = r[r["T"]==t]
        if len(match)>0:
            g = match["ecart"].values[0]
            ax.text(idx+xo[c], g+(0.01 if g>=0 else -0.03), f"{g:+.1%}", ha="center", fontsize=8)

plt.tight_layout()
path = IMG_DIR / "img_ecart_par_tranche.png"
plt.savefig(path, bbox_inches="tight")
print(f"Sauvegarde: {path}")
plt.show()


---

## 8. Validation

In [ ]:
print("VALIDATION")
v1 = np.all((probs_norm>=0)&(probs_norm<=1))
v2 = np.allclose(probs_norm.sum(axis=1), 1.0, atol=1e-6)
v3 = np.all(probs_raw.sum(axis=1)>1.0)
v4 = ll_norm > 0
v5 = 0 <= br_norm <= 2/3
v6 = df["match_id"].nunique() == len(df)
v7 = set(df["result_1n2"].unique()).issubset({"H","D","A"})
print(f"1. Probs 0-1: {v1}")
print(f"2. Somme=1: {v2}")
print(f"3. Marge>0: {v3}")
print(f"4. LogLoss>0: {v4}")
print(f"5. Brier valide: {v5} ({br_norm:.4f})")
print(f"6. Matchs uniques: {v6}")
print(f"7. Issues valides: {v7}")
print(f"Tout OK: {all([v1,v2,v3,v4,v5,v6,v7])}")

---

## 9. Resume des resultats

In [ ]:
baseline = {
    "model": "bookmaker_baseline_bet365",
    "odds_type": "closing",
    "season": "2024-2025",
    "n_matches": len(df),
    "log_loss_norm": float(ll_norm),
    "log_loss_raw": float(ll_raw),
    "brier_norm": float(br_norm),
    "brier_raw": float(br_raw),
    "moyenne_marge": float(margins.mean()),
    "mediane_marge": float(np.median(margins)),
    "random_log_loss": float(ll_random),
    "random_brier": float(br_random)
}

print("RESULTATS DE REFERENCE - BASELINE BET365")
print("=" * 50)
for k,v in baseline.items():
    print(f"  {k}: {v}")

rp = ROOT / "notebooks" / "baseline_results.json"
with open(rp, "w") as f:
    json.dump(baseline, f, indent=2)
print(f"Sauvegarde: {rp}")

---

## 10. Conclusion

- Log Loss et Brier plus bas = meilleur
- Calibration : pred vs obs doit etre proche de y=x
- Marge bet365 : ~5-6% en moyenne

### Limites
- Une saison (2024-2025)
- Un bookmaker (bet365)
- Cotes de fermeture (retrouvées)
- Descriptif, pas causal

### Prochaine étape : T11
Regression logistique multinomiale.